### Imports

In [4]:
from pathlib import Path
import sys

In [5]:

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(root_dir / "src"))
root_dir


PosixPath('/Volumes/256 SSD/deal_hunter')

In [6]:
from deal_hunter.config import settings

In [7]:
settings.rss_feed_url

['https://www.dealnews.com/c142/Electronics/?rss=1',
 'https://www.dealnews.com/c39/Computers/?rss=1',
 'https://www.dealnews.com/f1912/Smart-Home/?rss=1']

### Testing deals.py , pydantic models

In [5]:
from deal_hunter.models import ScrapedDeal, Deal,DealSelection, Opportunity

In [6]:
#making class
sd = ScrapedDeal(
    title = "Test Product" * 20,
    summary = "test SUmmary",
    url = "https://example.com",
    details = "some details" * 100,
    features = " Feature List" * 100
)



In [7]:
#checking methods
sd.truncate()
print(repr(sd))
print(sd.describe())
print(len(sd.title))
print(len(sd.details))

<Test ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest>
Title: Test ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest
Details: some detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome det
Features: Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature L

In [8]:
# Verifying other models
d = Deal(product_description="A widget", price=29.99, url="https://example.com")
ds = DealSelection(deals=[d])
o = Opportunity(deal=d, estimate=50.0, discount=0.40)
print(o)

deal=Deal(product_description='A widget', price=29.99, url='https://example.com') estimate=50.0 discount=0.4


In [ ]:
from deal_hunter.services import Rss_Service

rss = Rss_Service()
deals = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    max_per_feed=3,
)
print(f"Scraped {len(deals)} deals")
for d in deals[:3]:
    print(d.describe())
    print()

In [10]:
already_seen = {d.url for d in deals}
print(f"Known URLs: {len(already_seen)}")

deals_round2 = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    known_urls=already_seen,
    max_per_feed=3,
)
print(f"Round 2: {len(deals_round2)} new deals (should be 0 or very few)")

Known URLs: 9
Round 2: 0 new deals (should be 0 or very few)


### testing notification service

In [1]:
from deal_hunter.config import settings
from deal_hunter.services.notifications import PushoverNotifier

notifier = PushoverNotifier(
    user_key=settings.pushover_user,
    token=settings.pushover_token,
    url=settings.pushover_url,
)

print("user set?", bool(settings.pushover_user))
print("token set?", bool(settings.pushover_token))
print("url:", settings.pushover_url)

ModuleNotFoundError: No module named 'deal_hunter'

In [ ]:
ok = notifier.send("Phase 4 test from scanning.ipynb")
print("send() returned:", ok)

send() returned: True


In [ ]:
from deal_hunter.config import settings

print("user set?", bool(settings.pushover_user), "len:", len(settings.pushover_user))
print("token set?", bool(settings.pushover_token), "len:", len(settings.pushover_token))
print("url:", settings.pushover_url)


user set? True len: 30
token set? True len: 30
url: https://api.pushover.net/1/messages.json


In [ ]:
import requests
from deal_hunter.config import settings

payload = {
    "user": settings.pushover_user,
    "token": settings.pushover_token,
    "message": "debug test from notebook",
    "sound": "cashregister",
}

resp = requests.post(settings.pushover_url, data=payload, timeout=10)
print("HTTP status:", resp.status_code)
print("Response text:", resp.text)

HTTP status: 200
Response text: {"status":1,"request":"3f3f1595-0e96-47a6-9f10-6a1a0b8c16fd"}


### testing scanner agent

In [10]:
from deal_hunter.agents.scanner import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.test_scan()
print(result)

deals=[Deal(product_description='The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers 3840x2160 resolution, Dolby Vision HDR, and HDR10 support. It uses Roku OS for easy streaming app access and voice assistant compatibility. The TV also includes three HDMI ports for connecting multiple devices.', price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch 1080p personal meeting display designed for remote work. It includes a built-in 1080p webcam, stereo speakers, privacy shutter, and adjustable lighting features. It also supports wireless charging for compatible devices.', price=30.0, url='https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39'), Deal(product_description='The Lenovo IdeaPad Slim 5 features an AMD Ryzen 5 8645HS processor, 16GB RAM, and 512GB 